# 01. Synthetic Data Generator & Pipeline Ingestion
**Project:** Downtown Seattle Economic Vitality & MID Operations (2025–2026)
**Author:** Daniel Rodriguez III

This notebook contains the complete data engineering script to generate structurally sound synthetic datasets representing downtown Seattle's subdistricts, hourly pedestrian traffic, and Metropolitan Improvement District (MID) daily operations. It enforces relational integrity, primary key uniqueness (via collision checks), and exports raw assets for BigQuery ingestion.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import random
import os

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# 1. Dimension Table: Subdistricts
subdistricts = [
    {"subdistrict_id": "SD01", "name": "Retail Core", "mid_zone": "Central", "total_blocks": 45},
    {"subdistrict_id": "SD02", "name": "Belltown", "mid_zone": "North", "total_blocks": 55},
    {"subdistrict_id": "SD03", "name": "Pioneer Square", "mid_zone": "South", "total_blocks": 50},
    {"subdistrict_id": "SD04", "name": "Waterfront", "mid_zone": "West", "total_blocks": 35},
    {"subdistrict_id": "SD05", "name": "Denny Triangle", "mid_zone": "East", "total_blocks": 60},
    {"subdistrict_id": "SD06", "name": "Chinatown-ID", "mid_zone": "South-East", "total_blocks": 40}
]
df_subdistricts = pd.DataFrame(subdistricts)

# 2. Fact Table: Pedestrian Traffic (Hourly Jan 2025 - Oct 2026)
date_range = pd.date_range(start="2025-01-01", end="2026-10-15", freq="h")
pedestrian_data = []
for dt in date_range:
    for sd in subdistricts:
        base_count = 150 if sd["name"] == "Retail Core" else 80
        hour_factor = 2.5 if 11 <= dt.hour <= 18 else (0.3 if dt.hour <= 5 else 1.0)
        weekend_factor = 1.3 if dt.weekday() >= 5 and sd["name"] in ["Retail Core", "Waterfront"] else 1.0
        trend_factor = 1.0 + (dt - pd.Timestamp("2025-01-01")).days * 0.0003

        count = int(base_count * hour_factor * weekend_factor * trend_factor * np.random.uniform(0.8, 1.2))
        pedestrian_data.append({
            "timestamp": dt,
            "subdistrict_id": sd["subdistrict_id"],
            "sensor_id": f"SENS-{sd['subdistrict_id']}-01",
            "pedestrian_count": max(0, count)
        })
df_pedestrian = pd.DataFrame(pedestrian_data)

# 3. Fact Table: MID Operations Logs (Guaranteed Unique ticket_id Generation)
ops_services = ["Clean Team Bio-Sweeping", "Safety Escort", "Trash Removal (Tons)", "Business Contact", "Public Realm Activation"]
ops_data = []
generated_ticket_ids = set()
ops_dates = pd.date_range(start="2025-01-01", end="2026-10-15", freq="D")
for dt in ops_dates:
    for sd in subdistricts:
        n_tickets = np.random.randint(5, 25)
        for _ in range(n_tickets):
            while True:
                t_id = f"TKN-{random.randint(100000, 999999)}"
                if t_id not in generated_ticket_ids:
                    generated_ticket_ids.add(t_id)
                    break
            ops_data.append({
                "ticket_id": t_id,
                "date": dt.date(),
                "subdistrict_id": sd["subdistrict_id"],
                "service_category": random.choice(ops_services),
                "response_time_minutes": round(np.random.exponential(scale=15) + 2, 1),
                "resolution_status": random.choices(["Resolved", "Escalated", "Pending"], weights=[0.92, 0.05, 0.03])[0]
            })
df_ops = pd.DataFrame(ops_data)

# Integrity assertions
assert len(df_ops['ticket_id'].unique()) == len(df_ops), "Primary Key constraint violated!"
print(f"Generated {len(df_subdistricts)} subdistricts, {len(df_pedestrian)} pedestrian log rows, and {len(df_ops)} MID logs!")

In [ ]:
# Save outputs to content/data directories
os.makedirs("data/raw", exist_ok=True)
df_subdistricts.to_csv("data/raw/dim_subdistricts.csv", index=False)
df_pedestrian.to_csv("data/raw/fact_pedestrian_traffic.csv", index=False)
df_ops.to_csv("data/raw/fact_mid_operations.csv", index=False)
print("Raw CSV files saved to data/raw/")